## Example 1: Basic array expression evaluation

In [ ]:
# Example 1: Basic Array Expression Evaluation
import numexpr as ne
import numpy as np

print("Example 1: Basic numexpr array expression evaluation")

# Create test arrays
a = np.random.rand(100000).astype('float64')
b = np.random.rand(100000).astype('float64') 
c = np.random.rand(100000).astype('float64')

print(f"Array shapes: a={a.shape}, b={b.shape}, c={c.shape}")
print(f"Data type: {a.dtype}")

# Simple arithmetic expressions
result1 = ne.evaluate("a + b * c")
result2 = ne.evaluate("2.5 * a + 1.8")
result3 = ne.evaluate("where(a > 0.5, b, c)")

print(f"Expression 'a + b * c' result range: {result1.min():.4f} to {result1.max():.4f}")
print(f"Expression '2.5 * a + 1.8' result range: {result2.min():.4f} to {result2.max():.4f}")
print(f"Conditional 'where(a > 0.5, b, c)' selected {np.sum(a > 0.5)} elements from b")

# Test complex expression
complex_expr = "sqrt(a**2 + b**2) * sin(c * 3.14159)"
result4 = ne.evaluate(complex_expr)
print(f"Complex expression result range: {result4.min():.4f} to {result4.max():.4f}")

# Every expression must match the same computation done with numpy
assert np.allclose(result1, a + b * c)
assert np.allclose(result2, 2.5 * a + 1.8)
assert np.array_equal(result3, np.where(a > 0.5, b, c))
assert np.allclose(result4, np.sqrt(a**2 + b**2) * np.sin(c * 3.14159))

print("Basic expression evaluation completed successfully")

## Example 2: Performance comparison with NumPy

In [ ]:
# Example 2: Performance Comparison with NumPy
import numexpr as ne
import numpy as np
import time

print("Example 2: Performance comparison between numexpr and NumPy")

# Create large arrays for performance testing
size = 1000000
a = np.random.rand(size)
b = np.random.rand(size)
c = np.random.rand(size)

print(f"Testing with arrays of size: {size:,} elements")

# Expression to test: complex mathematical operation
expression = "2 * a + 3 * b**2 - sqrt(c + 0.1)"

# NumPy timing
start_time = time.time()
numpy_result = 2 * a + 3 * b**2 - np.sqrt(c + 0.1)
numpy_time = time.time() - start_time

# numexpr timing
start_time = time.time()
numexpr_result = ne.evaluate(expression)
numexpr_time = time.time() - start_time

# Verify results are equivalent
max_difference = np.max(np.abs(numpy_result - numexpr_result))
print(f"Maximum difference between results: {max_difference:.2e}")
assert np.allclose(numexpr_result, numpy_result), max_difference

# Performance comparison
speedup = numpy_time / numexpr_time
print(f"NumPy execution time: {numpy_time:.4f} seconds")
print(f"numexpr execution time: {numexpr_time:.4f} seconds")
print(f"Speedup: {speedup:.2f}x {'faster' if speedup > 1 else 'slower'}")

# Threading information
print(f"numexpr threads in use: {ne.nthreads}")
print(f"numexpr detected CPUs: {ne.detect_number_of_cores()}")

print("Performance comparison completed successfully")

## Example 3: Advanced mathematical functions and data types

In [ ]:
# Example 3: Advanced Mathematical Functions and Data Types
import numexpr as ne
import numpy as np

print("Example 3: Advanced mathematical functions and data type handling")

# Test different data types
data_types = [np.float32, np.float64, np.int32, np.int64]
array_size = 50000

for dtype in data_types:
    print(f"\nTesting with {dtype.__name__}:")
    
    if np.issubdtype(dtype, np.integer):
        arr = np.random.randint(1, 100, size=array_size, dtype=dtype)
        # Integer-safe operations
        result = ne.evaluate("arr**2 + arr * 3 + 5", local_dict={'arr': arr})
        print(f"  Integer expression: mean={np.mean(result):.2f}, max={np.max(result)}")
    else:
        arr = np.random.rand(array_size).astype(dtype)
        # Floating point mathematical functions
        result = ne.evaluate("exp(arr) * log(arr + 1) + arcsin(arr)", 
                           local_dict={'arr': arr})
        print(f"  Float expression: mean={np.mean(result):.4f}, std={np.std(result):.4f}")

# Complex mathematical expressions with constants
x = np.linspace(-2*np.pi, 2*np.pi, 10000)
y = np.random.rand(10000) * 0.1

# Trigonometric and hyperbolic functions
trig_result = ne.evaluate("sin(x) * cos(2*x) + tanh(x + y)")
print(f"\nTrigonometric expression result: min={trig_result.min():.4f}, max={trig_result.max():.4f}")

# Statistical and comparison operations
data = np.random.randn(100000)
mask_expr = "where((data > -1) & (data < 1), data**2, abs(data))"
masked_result = ne.evaluate(mask_expr, local_dict={'data': data})

print(f"Conditional masking: {np.sum((data > -1) & (data < 1))} elements squared")
print(f"Result statistics: mean={np.mean(masked_result):.4f}")

print("Mathematical functions testing completed successfully")

## Example 4: Multi-threading and memory optimization

In [ ]:
# Example 4: Multi-threading and Memory Optimization
import numexpr as ne
import numpy as np
import time

print("Example 4: Threading control and memory usage optimization")

# Create large array for memory testing
size = 2000000
data = np.random.rand(size)
print(f"Working with array size: {size:,} elements ({data.nbytes/1024**2:.1f} MB)")

# Test different thread configurations
thread_counts = [1, 2, 4]
expression = "sqrt(data**3 + exp(data) - log1p(data))"

results = {}
for nthreads in thread_counts:
    ne.set_num_threads(nthreads)
    print(f"\nTesting with {nthreads} thread(s):")
    
    # Warm-up run
    ne.evaluate(expression, local_dict={'data': data[:1000]})
    
    # Timed run
    start_time = time.time()
    result = ne.evaluate(expression, local_dict={'data': data})
    elapsed_time = time.time() - start_time
    
    results[nthreads] = elapsed_time
    throughput = data.nbytes / elapsed_time / 1024**2  # MB/s
    
    print(f"  Execution time: {elapsed_time:.4f} seconds")
    print(f"  Throughput: {throughput:.1f} MB/s")
    print(f"  Result range: {result.min():.4f} to {result.max():.4f}")

# Compare threading performance
best_time = min(results.values())
for threads, time_taken in results.items():
    efficiency = best_time / time_taken * 100
    print(f"{threads} thread(s): {efficiency:.1f}% efficiency")

# Memory usage optimization with in-place operations
large_array = np.random.rand(1000000)
print(f"\nMemory optimization test:")
print(f"Original array memory: {large_array.nbytes/1024**2:.1f} MB")

# In-place modification using numexpr
expected = large_array * 2 + 1
ne.evaluate("large_array * 2 + 1", out=large_array, local_dict={'large_array': large_array})
assert np.allclose(large_array, expected)
print(f"In-place operation completed - array modified directly")
print(f"Sample values after modification: {large_array[:5]}")

print("Threading and memory optimization testing completed successfully")

## Example 5: Real-world scientific computing applications

In [ ]:
# Example 5: Real-world Scientific Computing Applications
import numexpr as ne
import numpy as np

print("Example 5: Real-world scientific computing applications")

# Simulation: Financial option pricing (Black-Scholes-like calculation)
print("\nFinancial modeling: Option price calculation")
S = np.random.uniform(80, 120, 100000)  # Stock prices
K = 100.0  # Strike price
r = 0.05  # Risk-free rate
T = 1.0   # Time to maturity
sigma = 0.2  # Volatility

# Complex financial formula using numexpr
d1_expr = "(log(S/K) + (r + 0.5*sigma**2)*T) / (sigma*sqrt(T))"
d1 = ne.evaluate(d1_expr)

# Approximate normal CDF for demonstration
phi_d1_approx = ne.evaluate("0.5 * (1 + tanh(d1 * 0.7978845608))")

call_price_expr = "S * phi_d1_approx - K * exp(-r*T) * phi_d1_approx"
call_prices = ne.evaluate(call_price_expr, local_dict={'phi_d1_approx': phi_d1_approx})

print(f"Calculated {len(call_prices)} option prices")
print(f"Average call price: ${np.mean(call_prices):.2f}")
print(f"Price range: ${np.min(call_prices):.2f} to ${np.max(call_prices):.2f}")

# Physics simulation: Wave interference pattern
print("\nPhysics modeling: 2D wave interference")
x = np.linspace(-10, 10, 200)
y = np.linspace(-10, 10, 200)
X, Y = np.meshgrid(x, y)

# Two wave sources with different frequencies
wave_expr = "sin(sqrt((X-2)**2 + Y**2) * 0.8) + sin(sqrt((X+2)**2 + Y**2) * 1.2)"
wave_pattern = ne.evaluate(wave_expr)

interference_regions = ne.evaluate("where(abs(wave_pattern) < 0.2, 1, 0)")
constructive_regions = ne.evaluate("where(wave_pattern > 1.5, 1, 0)")

print(f"Wave pattern computed on {X.shape[0]}x{X.shape[1]} grid")
print(f"Interference nodes: {np.sum(interference_regions)} points")
print(f"Constructive regions: {np.sum(constructive_regions)} points")

# Data analysis: Statistical outlier detection
print("\nData analysis: Multi-dimensional outlier detection")
n_samples = 50000
data_x = np.random.normal(0, 1, n_samples)
data_y = np.random.normal(0, 1, n_samples)
data_z = np.random.normal(0, 1, n_samples)

# Add some outliers
outlier_indices = np.random.choice(n_samples, 200, replace=False)
data_x[outlier_indices] += np.random.uniform(3, 5, 200)

# Mahalanobis-like distance calculation
dist_expr = "sqrt(data_x**2 + data_y**2 + data_z**2)"
distances = ne.evaluate(dist_expr)
threshold = np.percentile(distances, 95)

outliers = ne.evaluate("where(distances > threshold, 1, 0)", 
                      local_dict={'threshold': threshold, 'distances': distances})

print(f"Detected {np.sum(outliers)} outliers from {n_samples} samples")
print(f"Detection threshold: {threshold:.3f}")

print("Scientific computing applications completed successfully")